# Twitter Topic Modeling
**Input:** `data/processed/twitter_processed/twitter_with_sentiment.csv`  
**Output:** `data/processed/twitter_processed/twitter_with_topics.csv` + figures

**Method:** Latent Dirichlet Allocation (LDA) via scikit-learn — same approach as the Reddit topic modeling notebook.

Analysis covered:
1. Text preprocessing → Bag of Words (TF-IDF)
2. Optimal K selection (perplexity curve)
3. LDA topic fitting
4. Top words per topic (bar charts)
5. Word clouds per topic
6. Topic distribution over time
7. Topic × Sentiment heatmap
8. Topic × Engagement heatmap
9. Pre/post ChatGPT topic shift

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from wordcloud import WordCloud

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

OUT_FIG  = os.path.abspath('../../reports/twitter_reports')
OUT_DATA = os.path.abspath('../../data/processed/twitter_processed')
os.makedirs(OUT_FIG,  exist_ok=True)
os.makedirs(OUT_DATA, exist_ok=True)
print('Setup OK')

## 1. Load Data

In [ ]:
df = pd.read_csv('../../data/processed/twitter_processed/twitter_with_sentiment.csv', parse_dates=['date'])
print(f'Loaded: {df.shape}')

# Use cleaned combined_text; fall back to text column
text_col = 'combined_text' if 'combined_text' in df.columns else 'cleaned_text'
texts = df[text_col].fillna('').tolist()
print(f'Text column: {text_col}')
print(f'Non-empty docs: {sum(1 for t in texts if len(t) > 5)}')

## 2. Vectorize Text
We use CountVectorizer (bag-of-words) for LDA. Twitter-specific stop words added.

In [ ]:
TWITTER_STOPWORDS = [
    'wikipedia', 'wiki', 'http', 'https', 'amp', 'rt', 'co',
    'im', 'dont', 'ive', 'youre', 'its', 'thats', 'just',
    'like', 'know', 'think', 'get', 'one', 'people', 'also',
    'really', 'even', 'still', 'would', 'could', 'said',
]

vectorizer = CountVectorizer(
    max_df        = 0.90,     # ignore terms in >90% docs (too common)
    min_df        = 5,        # ignore terms in <5 docs (too rare)
    stop_words    = 'english',
    max_features  = 3000,
    ngram_range   = (1, 2),   # unigrams + bigrams for richer topics
    token_pattern = r'[a-zA-Z]{3,}',  # only alpha tokens ≥3 chars
)

dtm = vectorizer.fit_transform(texts)
vocab = vectorizer.get_feature_names_out()
print(f'DTM shape: {dtm.shape}')
print(f'Vocabulary size: {len(vocab)}')

## 3. Select Optimal Number of Topics (K)
Train LDA for K = 3…10, compare perplexity. Lower perplexity = better fit.

In [ ]:
K_RANGE = range(3, 11)
perplexities = []

for k in K_RANGE:
    lda_k = LatentDirichletAllocation(
        n_components   = k,
        random_state   = SEED,
        max_iter       = 10,
        learning_method= 'online',
    )
    lda_k.fit(dtm)
    p = lda_k.perplexity(dtm)
    perplexities.append(p)
    print(f'  K={k:2d}  perplexity={p:.1f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_RANGE), perplexities, marker='o', color='#1565C0', lw=2)
ax.set_title('LDA Perplexity vs Number of Topics (K)', fontsize=13, weight='bold')
ax.set_xlabel('K (Number of Topics)')
ax.set_ylabel('Perplexity (lower = better)')
ax.set_xticks(list(K_RANGE))
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_tm_k_selection.png', dpi=150)
plt.show()

best_k = list(K_RANGE)[int(np.argmin(perplexities))]
print(f'\nBest K by perplexity: {best_k}')

## 4. Fit Final LDA Model
We use the K recommended by the perplexity curve. You can override `N_TOPICS` manually.

In [ ]:
N_TOPICS = best_k  # override here if desired, e.g. N_TOPICS = 7

lda = LatentDirichletAllocation(
    n_components    = N_TOPICS,
    random_state    = SEED,
    max_iter        = 30,
    learning_method = 'online',
    doc_topic_prior = 0.1,
    topic_word_prior= 0.01,
)
lda.fit(dtm)
print(f'Final LDA fit: K={N_TOPICS}, perplexity={lda.perplexity(dtm):.1f}')

# Assign dominant topic per tweet
doc_topics = lda.transform(dtm)
df['dominant_topic']    = doc_topics.argmax(axis=1)
df['topic_probability'] = doc_topics.max(axis=1)

## 5. Topic Labels — Top Words

In [ ]:
N_TOP_WORDS = 15

topic_keywords = {}
for i, component in enumerate(lda.components_):
    top_idx  = component.argsort()[::-1][:N_TOP_WORDS]
    top_words = [vocab[j] for j in top_idx]
    topic_keywords[i] = top_words
    print(f'Topic {i}: {" | ".join(top_words[:8])}')

# Manual short labels — edit these after reviewing the top words above
# They will update automatically on re-run
TOPIC_LABELS = {i: f'Topic {i}: {" / ".join(topic_keywords[i][:3])}' for i in range(N_TOPICS)}

## 6. Top Words Per Topic — Bar Charts

In [ ]:
n_cols = 3
n_rows = int(np.ceil(N_TOPICS / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5))
axes = axes.flatten()

palette = plt.cm.tab10.colors

for i, component in enumerate(lda.components_):
    top_idx   = component.argsort()[::-1][:10]
    top_words = [vocab[j] for j in top_idx]
    top_vals  = [component[j] for j in top_idx]

    ax = axes[i]
    ax.barh(top_words[::-1], top_vals[::-1], color=palette[i % 10], edgecolor='none')
    ax.set_title(f'Topic {i}', fontsize=11, weight='bold', color=palette[i % 10])
    ax.set_xlabel('Word Weight')

for j in range(N_TOPICS, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('LDA Topic Top Words — Twitter', fontsize=14, weight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_tm_top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Word Clouds Per Topic

In [ ]:
n_cols = 3
n_rows = int(np.ceil(N_TOPICS / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5))
axes = axes.flatten()

COLORMAP_CYCLE = ['Blues', 'Reds', 'Greens', 'Oranges', 'Purples',
                  'YlOrBr', 'PuBuGn', 'GnBu', 'OrRd', 'BuPu']

for i, component in enumerate(lda.components_):
    word_freq = {vocab[j]: float(component[j]) for j in range(len(vocab))}
    wc = WordCloud(
        width=400, height=250,
        background_color='white',
        colormap=COLORMAP_CYCLE[i % len(COLORMAP_CYCLE)],
        max_words=60,
        prefer_horizontal=0.9,
    ).generate_from_frequencies(word_freq)

    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].axis('off')
    axes[i].set_title(f'Topic {i}', fontsize=11, weight='bold',
                      color=plt.cm.tab10.colors[i % 10])

for j in range(N_TOPICS, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Topic Word Clouds — Twitter LDA', fontsize=14, weight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_tm_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Topic Distribution Overview

In [ ]:
topic_counts = df['dominant_topic'].value_counts().sort_index()
labels = [f'T{i}' for i in topic_counts.index]
colors = [plt.cm.tab10.colors[i % 10] for i in topic_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
axes[0].bar(labels, topic_counts.values, color=colors, edgecolor='none')
for j, (l, v) in enumerate(zip(labels, topic_counts.values)):
    axes[0].text(j, v + topic_counts.max()*0.01, str(v), ha='center', fontsize=9)
axes[0].set_title('Tweet Count by Dominant Topic', fontsize=12, weight='bold')
axes[0].set_xlabel('Topic')
axes[0].set_ylabel('Count')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    topic_counts.values,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    pctdistance=0.82
)
axes[1].set_title('Topic Share of Voice', fontsize=12, weight='bold')

plt.suptitle('LDA Topic Distribution', fontsize=13, weight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_tm_topic_overview.png', dpi=150)
plt.show()

## 9. Topic Trend Over Time

In [ ]:
df['month'] = df['date'].dt.to_period('M').dt.to_timestamp()

topic_monthly = (
    df.groupby(['month', 'dominant_topic'])
    .size()
    .unstack(fill_value=0)
)

# Normalise to % per month
topic_monthly_pct = topic_monthly.div(topic_monthly.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 5))
colors_t = [plt.cm.tab10.colors[i % 10] for i in topic_monthly_pct.columns]

ax.stackplot(
    topic_monthly_pct.index,
    [topic_monthly_pct[c] for c in topic_monthly_pct.columns],
    labels=[f'Topic {c}' for c in topic_monthly_pct.columns],
    colors=colors_t,
    alpha=0.85
)
ax.axvline(pd.Timestamp('2022-11-30'), color='black', ls='--', lw=1.5, label='ChatGPT launch')
ax.set_title('Topic Share Over Time — Twitter', fontsize=13, weight='bold')
ax.set_ylabel('% of Monthly Tweets')
ax.set_xlabel('Month')
ax.set_ylim(0, 100)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_tm_topic_trend.png', dpi=150)
plt.show()

## 10. Topic × Sentiment Heatmap

In [ ]:
sent_col = 'roberta_label' if 'roberta_label' in df.columns else 'vader_label'

pivot = (
    df.groupby(['dominant_topic', sent_col])
    .size()
    .unstack(fill_value=0)
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct.index = [f'Topic {i}' for i in pivot_pct.index]

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    pivot_pct[['positive','neutral','negative']].T,
    ax=ax, cmap='RdYlGn', annot=True, fmt='.1f',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': '% of Topic Tweets'}
)
ax.set_title('Sentiment % per Topic (RoBERTa)', fontsize=13, weight='bold')
ax.set_xlabel('Topic')
ax.set_ylabel('Sentiment')
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_tm_sentiment_heatmap.png', dpi=150)
plt.show()

## 11. Topic × Engagement Heatmap

In [ ]:
eng_pivot = (
    df.groupby('dominant_topic')[['likes','retweets','replies','total_engagement']]
    .mean()
    .round(2)
)
eng_pivot.index = [f'Topic {i}' for i in eng_pivot.index]

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    eng_pivot.T,
    ax=ax, cmap='Blues', annot=True, fmt='.1f',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Average Count'}
)
ax.set_title('Average Engagement per Topic', fontsize=13, weight='bold')
ax.set_xlabel('Topic')
ax.set_ylabel('Metric')
plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_tm_engagement_heatmap.png', dpi=150)
plt.show()

print('\nTop topic by engagement:')
print(eng_pivot.sort_values('total_engagement', ascending=False).head(3))

## 12. Pre/Post ChatGPT Topic Shift

In [ ]:
if 'period' not in df.columns:
    chatgpt = pd.Timestamp('2022-11-30')
    df['period'] = df['date'].apply(lambda d: 'post-ChatGPT' if d >= chatgpt else 'pre-ChatGPT')

period_topics = (
    df.groupby(['period', 'dominant_topic'])
    .size()
    .unstack(fill_value=0)
)
period_pct = period_topics.div(period_topics.sum(axis=1), axis=0) * 100
period_pct.columns = [f'Topic {c}' for c in period_pct.columns]

fig, ax = plt.subplots(figsize=(10, 4))
period_pct.reindex(['pre-ChatGPT', 'post-ChatGPT']).T.plot(
    kind='bar', ax=ax,
    color=['#42A5F5', '#EF5350'],
    edgecolor='none', width=0.6
)
ax.set_title('Topic Share — Pre vs Post ChatGPT Launch', fontsize=13, weight='bold')
ax.set_ylabel('% of Period Tweets')
ax.set_xlabel('Topic')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30)
ax.legend(title='Period', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUT_FIG}/tw_tm_pre_post_shift.png', dpi=150)
plt.show()

## 13. Topic Summary Table

In [ ]:
summary = pd.DataFrame({
    'topic_id'      : range(N_TOPICS),
    'top_words'     : [' | '.join(topic_keywords[i][:6]) for i in range(N_TOPICS)],
    'tweet_count'   : [int(topic_counts.get(i, 0)) for i in range(N_TOPICS)],
    'pct_total'     : [(topic_counts.get(i, 0) / len(df) * 100).round(2) for i in range(N_TOPICS)],
})
print(summary.to_string(index=False))

summary.to_csv(os.path.join(OUT_DATA, 'twitter_topic_summary.csv'), index=False)
print('\nSaved topic_summary.csv')

## 14. Save Topic-Tagged Dataset

In [ ]:
out = os.path.join(OUT_DATA, 'twitter_with_topics.csv')
df.to_csv(out, index=False)
print(f'Saved → {out}')
print(f'Shape: {df.shape}')